# Challenge the movement interpretation with controls

This notebook connects observation perturbations, recording conditions, source support, and external reference shift to the limits of a physiological interpretation.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p/'worldmodels/v2_data.py').is_file()), None)
if PROJECT is None:
    PROJECT = next(p/'experiments/aaai-worldmodels' for p in candidates if (p/'experiments/aaai-worldmodels/worldmodels/v2_data.py').is_file())
sys.path.insert(0, str(PROJECT))
REPOSITORY = PROJECT.parents[1]
from worldmodels.v2_data import load_cohort
from worldmodels import v2_evaluation as ev
from worldmodels import v2_models as models
from worldmodels.v2_runner import window_errors, verify_freeze
COHORT = PROJECT/'artifacts/runs/reference-v2-cohort-v1'
RUN = PROJECT/'artifacts/runs/reference-v2-core-v1'
TRANSFER = PROJECT/'artifacts/runs/reference-v2-transfer-v1'
FIGURES = PROJECT/'artifacts/executed-reference-v2/figures'
FIGURES.mkdir(parents=True, exist_ok=True)
USED_INPUTS={}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def record_input(path):
    path=Path(path).resolve()
    key=str(path.relative_to(REPOSITORY)) if path.is_relative_to(REPOSITORY) else str(path)
    USED_INPUTS[key]=sha(path)
def read_json(path):
    path = Path(path)
    assert path.is_file(), f'Required completed artifact is absent: {path}. Read docs/reference_v2/README.md for prerequisites.'
    record_input(path)
    return json.loads(path.read_text())
def read_csv(path, **kwargs):
    record_input(path)
    return pd.read_csv(path, **kwargs)
def artifact(name, directory=RUN):
    path = directory/name
    manifest = read_json(directory/'artifact_hashes.json')
    assert name in manifest and path.is_file(), f'Unrecorded or missing artifact: {path}'
    assert sha(path) == manifest[name], f'Artifact hash differs from the completed run: {path}'
    record_input(path)
    return path
def completed_run():
    result = read_json(artifact('results.json'))
    assert result['status']=='complete' and not result.get('test_only', False)
    return result
def save_figure(figure, name):
    figure.savefig(FIGURES/(name+'.svg'), bbox_inches='tight')
    figure.savefig(FIGURES/(name+'.png'), dpi=150, bbox_inches='tight')
freeze = verify_freeze()
record_input(PROJECT/'docs/reference_v2/protocol_freeze.json')
x, valid, metadata, actual_times = load_cohort(COHORT, verify=True)
cohort_identity=read_json(COHORT/'identity.json');record_input(COHORT/'cohort.csv')
for name in cohort_identity['prepared_inputs']:
    record_input(REPOSITORY/cohort_identity['prepared_directory']/name)
assert x.shape == (len(metadata),24,16,2) and len(metadata)==2815
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'svg.fonttype':'none','axes.spines.top':False,'axes.spines.right':False})
print('Verified real observations:',len(metadata),'windows;',metadata.source_id.nunique(),'source videos.')
print('Observation units: pelvis-relative 2D coordinates divided by a prefix torso scale. Times are decoder seconds.')


## Check the fixed observation perturbations

The robustness analysis changes declared prefix observations while holding future targets fixed. It measures sensitivity to observation loss, noise, or time-order changes. These are information controls and do not simulate a neurological lesion or treatment.

In [ ]:
result=completed_run()
ROBUSTNESS=PROJECT/'artifacts/runs/reference-v2-robustness-v1'
robustness=read_json(artifact('results.json',ROBUSTNESS))
display(pd.DataFrame(robustness['normal_forecast_contrasts']))
print('Read each perturbation together with its retained support and definition. A changed input asks a different information-access question.')


## Compare recording views and observation quality

Camera view changes projected articulation. Pose confidence and missingness can also alter which motion is visible. View or quality slices must therefore show source counts and their changed population. A sparse slice cannot establish the same conclusion as the complete cohort.

In [ ]:
source=read_csv(artifact('source_anomaly_scores.csv'))
display(read_csv(COHORT/'coverage_by_view.csv'))
display(robustness['nuisance_sensitivity'])
fig,axes=plt.subplots(1,2,figsize=(11,4.8))
for label,color in [('normal','#197c80'),('other','#ad632d')]:
    keep=source.label.eq('normal') if label=='normal' else source.label.ne('normal')
    name=('Normal annotation' if label=='normal' else 'Abnormal annotations')+f' ({int(keep.sum())} source videos)'
    axes[0].scatter(source.loc[keep,'prefix_motion'],source.loc[keep,'calibrated_two_sided'],color=color,label=name,alpha=.75)
    axes[1].scatter(source.loc[keep,'prefix_missingness'],source.loc[keep,'calibrated_two_sided'],color=color,alpha=.75)
axes[0].set(xlabel='Observed prefix motion, normalized coordinates per second',ylabel='Calibrated two-sided score',title='Motion remains a competing explanation.')
axes[1].set(xlabel='Prefix missingness fraction',ylabel='Calibrated two-sided score',title='Observation quality remains a competing explanation.')
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',ncol=2,frameon=False)
fig.tight_layout(rect=[0,.12,1,1]);save_figure(fig,'25_motion_quality_controls');plt.show()


## Inspect external reference extrapolation

The nuisance reference was fitted on a small normal-source population. Different acquisition conditions can place external observations outside that population's motion or missingness range. Scores at an empirical-reference endpoint are coarse ranks, not evidence of a clinically calibrated probability.

In [ ]:
transfer=read_json(artifact('results.json',TRANSFER))
display(transfer['covariate_shift'])
participants=read_csv(artifact('participant_scores.csv',TRANSFER))
display(participants.groupby('label').agg(participants=('source_id','size'),motion_median=('prefix_motion','median'),missingness_median=('prefix_missingness','median')))
print('The frozen reference is reported without fitting a new transform to improve the external labels.')


## Connect physiology to observations we actually measured

Bilateral timing, muscle coordination, and compensatory movement motivate useful questions. Our measurements are projected joint positions and their forecasts. We therefore test movement relationships and alternative explanations, while treating the physiological cause of an individual score as unmeasured.

In [ ]:
display(pd.DataFrame([
    {'motivation':'Bilateral timing may be altered.','observed_test':'Compare aligned and shifted histories.','limit':'A timing perturbation is not a neural intervention.'},
    {'motivation':'Compensation can be repeatable.','observed_test':'Compare raw error, low motion, and both normal-reference tails.','limit':'Easy prediction does not establish healthy movement.'},
    {'motivation':'Several joints move together.','observed_test':'Compare anatomical tubes with matched random joint groups.','limit':'An anatomical mask is not a measurement of muscle synergies.'},
    {'motivation':'Movement differs across clinical annotations.','observed_test':'Show condition profiles and shared normal references.','limit':'Folder labels do not independently confirm diagnosis.'},
]))
print('Reliable phase, learned-mask, latent-objective, and stochastic extensions require separately declared gate decisions.')
display(robustness['source_identity_probe'])


## Read every limitation at the correct unit

Many windows improve the description of a recording, but they do not create more independent people. Condition slices share normal references; altered observation support changes the population. The internal cohort and external transfer answer complementary questions, and neither alone supplies a treatment-response model.

In [ ]:
display(result['limitations'])
print('Internal sources:',result['sources'],'retained normal sources:',result['normal_sources'])
print('The output README states the remaining participant, site, label, and clinical validation requirements.')

# Retain the exact inputs used by this independent-kernel execution.
input_path=PROJECT/'artifacts/executed-reference-v2/inputs/25_robustness_and_movement_interpretation.json'
input_path.parent.mkdir(parents=True,exist_ok=True)
input_path.write_text(json.dumps({'path_base':'repository root','inputs':USED_INPUTS},indent=2)+'\n')
